In [ ]:
pip install tensorflow


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2_as_graph

def get_flops(model: Model, batch_size: int = 1) -> int:
    """
    Calculate FLOPs for a tf.keras.Model or tf.keras.Sequential model.

    Args:
        model (Model): The Keras model instance.
        batch_size (int): Batch size to use for the calculation.

    Returns:
        int: Total number of FLOPs.
    """
    # Ensure the model is built
    if not model.built:
        raise ValueError("The model must be built before computing FLOPs.")

    # Create input signature
    input_signature = []
    for input_tensor in model.inputs:
        input_shape = [batch_size] + list(input_tensor.shape[1:])
        input_signature.append(tf.TensorSpec(shape=input_shape, dtype=input_tensor.dtype))

    # Convert Keras model to ConcreteFunction
    @tf.function
    def model_forward(*args):
        return model(*args)

    concrete_func = model_forward.get_concrete_function(*input_signature)

    # Convert variables to constants
    frozen_func, _ = convert_variables_to_constants_v2_as_graph(concrete_func)

    # Calculate FLOPs with tf.profiler
    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

    flops = tf.compat.v1.profiler.profile(graph=frozen_func.graph,
                                          run_meta=run_meta, cmd='op', options=opts)

    return flops.total_float_ops




# **My Model**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D, BatchNormalization, AveragePooling2D, GlobalAveragePooling2D, Dense, Dropout, Input

# Define your model
model = Sequential([
    Input(shape=(51, 51, 3)),
    Conv2D(32, kernel_size=(3, 3), strides=(2, 2), padding="same", activation='relu'),
    BatchNormalization(),
    DepthwiseConv2D(kernel_size=(3, 3), padding="same", activation='relu'),
    BatchNormalization(),
    Conv2D(48, kernel_size=(1, 1), activation='relu'),
    BatchNormalization(),
    AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),
    DepthwiseConv2D(kernel_size=(3, 3), padding="same", activation='relu'),
    BatchNormalization(),
    Conv2D(96, kernel_size=(1, 1), activation='relu'),
    BatchNormalization(),
    AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),
    Dropout(0.2),
    DepthwiseConv2D(kernel_size=(3, 3), padding="same", activation='relu'),
    BatchNormalization(),
    Conv2D(128, kernel_size=(1, 1), activation='relu'),
    BatchNormalization(),
    AveragePooling2D(pool_size=(2, 2), strides=(2, 2)),
    Dropout(0.2),
    DepthwiseConv2D(kernel_size=(3, 3), padding="same", activation='relu'),
    BatchNormalization(),
    Conv2D(128, kernel_size=(1, 1), activation='relu'),
    BatchNormalization(),
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(6, activation='softmax')
])

# Ensure the model is built
model.build(input_shape=(None, 51, 51, 3))

# Now compute FLOPs
flops = get_flops(model, batch_size=1)
mops = flops / 1_000_000  # Convert to MOps

print(f"FLOPs: {flops}")
print(f"MOps: {mops:.2f}")


FLOPs: 7008980
MOps: 7.01


# Paper2

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

IMG_DIM = 51
NUM_CLASSES = 6

def dilated_deformable_conv(x, filters, kernel_size, dilation_rate=1):
    conv = layers.Conv2D(filters, kernel_size, dilation_rate=dilation_rate, padding='same', activation=None)(x)
    bn = layers.BatchNormalization()(conv)
    activation = layers.Activation('swish')(bn)
    return activation

def depthwise_conv(x, filters, kernel_size, strides=1):
    depthwise_conv_layer = layers.DepthwiseConv2D(kernel_size, strides=strides, padding='same', activation=None)(x)
    bn = layers.BatchNormalization()(depthwise_conv_layer)
    activation = layers.Activation('swish')(bn)
    conv = layers.Conv2D(filters, (1, 1), padding='same', activation='swish')(activation)
    return conv

def create_model():
    input_layer = layers.Input(shape=(IMG_DIM, IMG_DIM, 3))
    x = layers.Conv2D(32, (3, 3), padding='same', activation='swish')(input_layer)
    x = dilated_deformable_conv(x, 64, (3, 3), dilation_rate=1)
    x = dilated_deformable_conv(x, 128, (3, 3), dilation_rate=2)
    x = dilated_deformable_conv(x, 256, (3, 3), dilation_rate=4)
    x = layers.MaxPooling2D(pool_size=(5, 5), strides=1, padding='same')(x)
    x = layers.MaxPooling2D(pool_size=(9, 9), strides=1, padding='same')(x)
    x = layers.MaxPooling2D(pool_size=(13, 13), strides=1, padding='same')(x)
    x = depthwise_conv(x, 128, (3, 3))
    x = depthwise_conv(x, 64, (3, 3))
    x = depthwise_conv(x, 32, (3, 3))
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.6)(x)
    output_layer = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output_layer)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model2

# Create and build the model
model2 = create_model()
model2.build(input_shape=(1, IMG_DIM, IMG_DIM, 3))  # Important: Build the model before computing FLOPs

# Now safely compute FLOPs using your existing get_flops method
flops = get_flops(model2, batch_size=1)
mops = flops / 1_000_000

print(f"FLOPs: {flops}")
print(f"MOps: {mops:.2f}")


FLOPs: 2535542756
MOps: 2535.54


# **Paper1**

In [ ]:
from tensorflow import keras

IMG_DIM = 51
NUM_CLASSES = 6

def create_model2():
    """
    Create a sequential model with depthwise separable convolutions
    and fully connected layers for image classification.

    Returns:
        keras.Sequential: Compiled neural network model
    """
    model2 = keras.Sequential([
        # First Convolution layer
        keras.layers.Conv2D(64, kernel_size=(3, 3), strides=(1, 1), padding="same", activation='relu', input_shape=(IMG_DIM, IMG_DIM, 3)),
        keras.layers.MaxPooling2D(pool_size=(2, 2)),
        keras.layers.Dropout(0.25),

        # Second Convolution layer
        keras.layers.Conv2D(64, kernel_size=(3, 3), strides=(1, 1), padding="same", activation='relu'),
        keras.layers.MaxPooling2D(pool_size=(2, 2)),
        keras.layers.Dropout(0.25),

        # Third Convolution layer
        keras.layers.Conv2D(128, kernel_size=(3, 3), strides=(1, 1), padding="same", activation='relu'),
        keras.layers.MaxPooling2D(pool_size=(2, 2)),
        keras.layers.Dropout(0.25),

        # Fourth Convolution layer
        keras.layers.Conv2D(128, kernel_size=(3, 3), strides=(1, 1), padding="same", activation='relu'),
        keras.layers.MaxPooling2D(pool_size=(2, 2)),
        keras.layers.Dropout(0.25),

        # Fifth Convolution layer
        keras.layers.Conv2D(256, kernel_size=(3, 3), strides=(1, 1), padding="same", activation='relu'),
        keras.layers.MaxPooling2D(pool_size=(2, 2)),
        keras.layers.Dropout(0.25),

        # Global Average Pooling
        keras.layers.GlobalAveragePooling2D(),

        # Fully connected layer
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.5),

        # Output layer (6 classes)
        keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    # Compile the model
    model1.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model1
model1 = create_model2()
model1.build(input_shape=(None, IMG_DIM, IMG_DIM, 3))  # Ensure model is built
flops = get_flops(model1, batch_size=1)
mops = flops / 1_000_000

print(f"FLOPs: {flops}")
print(f"MOps: {mops:.2f}")


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


FLOPs: 92748196
MOps: 92.75


# **Paper3**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_DIM = 51
NUM_CLASSES = 6

# Define RepVGG Block
class RepVGGBlock(tf.keras.layers.Layer):
    def __init__(self, filters, kernel_size=3, stride=1, use_identity=True):
        super(RepVGGBlock, self).__init__()
        self.conv3x3 = layers.Conv2D(filters, kernel_size, strides=stride, padding='same', use_bias=False)
        self.bn3x3 = layers.BatchNormalization()
        self.conv1x1 = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)
        self.bn1x1 = layers.BatchNormalization()

        if use_identity:
            if stride > 1:
                self.identity = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False)
            else:
                self.identity = layers.BatchNormalization()
        else:
            self.identity = None

        self.relu = layers.ReLU()

    def call(self, inputs, training=False):
        x1 = self.bn3x3(self.conv3x3(inputs), training=training)
        x2 = self.bn1x1(self.conv1x1(inputs), training=training)

        if self.identity is not None:
            x3 = self.identity(inputs)
            out = x1 + x2 + x3
        else:
            out = x1 + x2
        return self.relu(out)

# Define MSCA Module
class MSCA_Module(tf.keras.layers.Layer):
    def __init__(self, filters):
        super(MSCA_Module, self).__init__()
        self.conv1 = layers.Conv2D(filters // 2, kernel_size=1, activation='relu', padding='same')
        self.conv3 = layers.Conv2D(filters // 2, kernel_size=3, activation='relu', padding='same')
        self.conv5 = layers.Conv2D(filters, kernel_size=5, activation='relu', padding='same')
        self.bn = layers.BatchNormalization()

    def call(self, inputs, training=False):
        x1 = self.conv1(inputs)
        x2 = self.conv3(inputs)
        x3 = self.conv5(inputs)
        x = tf.concat([x1, x2, x3], axis=-1)
        x = self.bn(x, training=training)
        return x

# Define the model
def create_model3():
    inputs = layers.Input(shape=(IMG_DIM, IMG_DIM, 3))

    x = RepVGGBlock(64, stride=2)(inputs)    # 26x26x64
    x = RepVGGBlock(128, stride=2)(x)        # 13x13x128
    x = RepVGGBlock(256, stride=2)(x)        # 7x7x256

    x = MSCA_Module(256)(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model3 = models.Model(inputs, outputs)
    model3.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model3
model3 = create_model3()
model3.build(input_shape=(None, IMG_DIM, IMG_DIM, 3))  # Ensure it's built before profiling
flops = get_flops(model3, batch_size=1)
mops = flops / 1_000_000

print(f"FLOPs: {flops}")
print(f"MOps: {mops:.2f}")


FLOPs: 261956612
MOps: 261.96


# **Paper4**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_DIM = 51
NUM_CLASSES = 6

# GSConv Layer
class GSConv(tf.keras.layers.Layer):
    def __init__(self, out_channels):
        super(GSConv, self).__init__()
        self.conv1 = layers.Conv2D(out_channels // 2, kernel_size=1, padding='same', activation='relu')
        self.conv2 = layers.Conv2D(out_channels // 2, kernel_size=3, padding='same', activation='relu')
        self.shuffle = layers.Lambda(lambda x: tf.experimental.numpy.moveaxis(x, 1, -1))  # simple channel shuffle

    def call(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x)
        out = tf.concat([x1, x2], axis=-1)
        return self.shuffle(out)

# GRSCPSELAN Layer (Residual Block)
class GRSCPSELAN(tf.keras.layers.Layer):
    def __init__(self, filters):
        super(GRSCPSELAN, self).__init__()
        self.conv1 = layers.Conv2D(filters, 3, padding='same', activation='relu')
        self.conv2 = layers.Conv2D(filters, 3, padding='same', activation='relu')
        self.shortcut = layers.Conv2D(filters, 1, padding='same')
        self.bn = layers.BatchNormalization()
        self.relu = layers.ReLU()

    def call(self, x, training=False):
        residual = self.shortcut(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn(x + residual, training=training)
        return self.relu(x)

# Model Definition
def create_model4():
    inputs = layers.Input(shape=(IMG_DIM, IMG_DIM, 3))

    x = layers.Conv2D(32, 3, strides=1, padding='same', activation='relu')(inputs)
    x = GRSCPSELAN(64)(x)
    x = layers.MaxPooling2D()(x)
    x = GRSCPSELAN(128)(x)
    x = layers.MaxPooling2D()(x)
    x = GRSCPSELAN(256)(x)

    x = GSConv(256)(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model4 = models.Model(inputs, outputs)
    model4.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model4
model4 = create_model4()
model4.build(input_shape=(1, IMG_DIM, IMG_DIM, 3))  # Build the model explicitly
flops = get_flops(model4, batch_size=1)
mops = flops / 1_000_000

print(f"FLOPs: {flops}")
print(f"MOps: {mops:.2f}")


FLOPs: 950230500
MOps: 950.23


# **Paper5**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

IMG_DIM = 51
NUM_CLASSES = 6

# Conv → BN → ReLU block
def conv_bn_relu(x, filters, kernel_size, stride=1):
    x = layers.Conv2D(filters, kernel_size, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

# Multi-Scale Depthwise Block
def multi_block(x, filters, stride=1):
    d3 = layers.DepthwiseConv2D(3, strides=stride, padding='same', use_bias=False)(x)
    d5 = layers.DepthwiseConv2D(5, strides=stride, padding='same', use_bias=False)(x)
    d7 = layers.DepthwiseConv2D(7, strides=stride, padding='same', use_bias=False)(x)

    d3 = layers.BatchNormalization()(d3)
    d5 = layers.BatchNormalization()(d5)
    d7 = layers.BatchNormalization()(d7)

    d3 = layers.ReLU()(d3)
    d5 = layers.ReLU()(d5)
    d7 = layers.ReLU()(d7)

    concat = layers.Concatenate()([d3, d5, d7])
    pw = layers.Conv2D(filters, kernel_size=1, padding='same', use_bias=False)(concat)
    pw = layers.BatchNormalization()(pw)
    pw = layers.ReLU()(pw)
    return pw

# Model Builder
def create_model5():
    input_layer = layers.Input(shape=(IMG_DIM, IMG_DIM, 3))

    x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(input_layer)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = multi_block(x, 64, stride=2)
    x = multi_block(x, 96, stride=2)
    x = multi_block(x, 128, stride=2)
    x = multi_block(x, 128, stride=1)
    x = multi_block(x, 192, stride=1)

    x = layers.Conv2D(192, 1, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Dropout(0.6)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.7)(x)
    output = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model5 = Model(inputs=input_layer, outputs=output)

    model5.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model5
model5 = create_model5()
model5.build(input_shape=(1, IMG_DIM, IMG_DIM, 3))
flops = get_flops(model5, batch_size=1)
mops = flops / 1_000_000

print(f"FLOPs: {flops}")
print(f"MOps: {mops:.2f}")


FLOPs: 13894116
MOps: 13.89
